In [ ]:
import h5py
import numpy as np
from pathlib import Path

np.random.seed(42)

# Paths relative to repo root
LOGS_ROOT = Path("../../logs/wicompass")
KNN_ROOT = LOGS_ROOT / "knn_coverage"
TOKEN_ROOT = LOGS_ROOT / "encoded_tokens"

N_SAMPLES = 10
K = 10  # use k=10 coverage results

# 1. Sample 10 random poses from MMBody dataset
with h5py.File(KNN_ROOT / f'mmbody/k{K}_dedup.h5', 'r') as f:
    B_raw = f['/processed/raw/B'][:]
    rand_idx = np.random.choice(len(B_raw), size=N_SAMPLES, replace=False)
    mmbody_tokens = B_raw[rand_idx]
print(f"MMBody tokens: {mmbody_tokens.shape}")

# 2. Sample 10 random poses from MMFi dataset
with h5py.File(KNN_ROOT / f'mmfi/k{K}_dedup.h5', 'r') as f:
    B_raw = f['/processed/raw/B'][:]
    rand_idx = np.random.choice(len(B_raw), size=N_SAMPLES, replace=False)
    mmfi_tokens = B_raw[rand_idx]
print(f"MMFi tokens: {mmfi_tokens.shape}")

# 3. Sample 10 poses in AMASS but NOT covered by MMBody
with h5py.File(KNN_ROOT / f'mmbody/k{K}_dedup.h5', 'r') as f:
    A_raw = f['/processed/raw/A'][:]
    uncovered_idx = f['/coverage_indices/A_uncovered_by_B'][:]
    A_uncovered = A_raw[uncovered_idx]
    rand_idx = np.random.choice(len(A_uncovered), size=N_SAMPLES, replace=False)
    mmbody_uncovered_tokens = A_uncovered[rand_idx]
print(f"AMASS uncovered by MMBody: {mmbody_uncovered_tokens.shape} (from {len(uncovered_idx)} uncovered)")

# 4. Sample 10 poses in AMASS but NOT covered by MMFi
with h5py.File(KNN_ROOT / f'mmfi/k{K}_dedup.h5', 'r') as f:
    A_raw = f['/processed/raw/A'][:]
    uncovered_idx = f['/coverage_indices/A_uncovered_by_B'][:]
    A_uncovered = A_raw[uncovered_idx]
    rand_idx = np.random.choice(len(A_uncovered), size=N_SAMPLES, replace=False)
    mmfi_uncovered_tokens = A_uncovered[rand_idx]
print(f"AMASS uncovered by MMFi: {mmfi_uncovered_tokens.shape} (from {len(uncovered_idx)} uncovered)")


In [ ]:
import sys
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Add src to path so wicompass package is importable
repo_root = Path("../..").resolve()
sys.path.insert(0, str(repo_root / "src"))

from wicompass.evaluation.core import load_config, load_model
from wicompass.visualization import plot_single_pose, save_pose_plot

# Load VQ-VAE model for decoding tokens -> poses
config_path = repo_root / "src/wicompass/configs/joint_vae_base_tokennum16_tokenclass64.json"
model_path = repo_root / "logs/vqvae/vqvae_tokennum16_tokenclass64/best_model.pth"

config = load_config(str(config_path))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_model(config['model'], str(model_path), device)
model.eval()
print(f"Model loaded on {device}, codebook shape: {model.codebook.shape}")


def tokens_to_poses(tokens, model, device='cuda'):
    """Decode token indices -> (N, num_joints, 3) joint positions via VQ-VAE."""
    with torch.no_grad():
        tokens_tensor = torch.from_numpy(tokens).long().to(device)
        quantized = F.embedding(tokens_tensor, model.codebook)
        poses = model.decode(quantized)
    return poses.cpu().numpy()


# Convert all 4 groups of tokens to poses
mmbody_poses = tokens_to_poses(mmbody_tokens, model, device)
mmfi_poses = tokens_to_poses(mmfi_tokens, model, device)
mmbody_uncovered_poses = tokens_to_poses(mmbody_uncovered_tokens, model, device)
mmfi_uncovered_poses = tokens_to_poses(mmfi_uncovered_tokens, model, device)

print(f"MMBody poses: {mmbody_poses.shape}")
print(f"MMFi poses: {mmfi_poses.shape}")
print(f"AMASS uncovered by MMBody poses: {mmbody_uncovered_poses.shape}")
print(f"AMASS uncovered by MMFi poses: {mmfi_uncovered_poses.shape}")

In [ ]:
import io
from PIL import Image
from wicompass.visualization.constants import BONE_CONNECTIONS, BODY_PART_COLORS, BONE_PART_MAPPING


def _render_single_pose(joints, elev=10, azim=0, dpi=200,
                        joint_size=30, line_width=3.0, padding=0.05):
    """Render one pose to a PIL Image with white background."""
    fig = plt.figure(figsize=(3, 3))
    ax = fig.add_subplot(111, projection='3d')

    # Draw bones
    for j1, j2 in BONE_CONNECTIONS:
        if j1 >= len(joints) or j2 >= len(joints):
            continue
        part = BONE_PART_MAPPING.get((j1, j2), 'spine')
        color = BODY_PART_COLORS[part]
        ax.plot([joints[j1, 0], joints[j2, 0]],
                [joints[j1, 1], joints[j2, 1]],
                [joints[j1, 2], joints[j2, 2]],
                color=color, linewidth=line_width, alpha=0.85, solid_capstyle='round')

    # Draw joints
    ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2],
               c='#2c3e50', s=joint_size, alpha=0.9,
               edgecolors='white', linewidths=0.8, zorder=5)

    # Tight equal-aspect limits
    ranges = joints.max(axis=0) - joints.min(axis=0)
    max_range = ranges.max() / 2.0
    pad = max_range * padding
    mid = (joints.max(axis=0) + joints.min(axis=0)) * 0.5
    for setter, c in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], mid):
        setter(c - max_range - pad, c + max_range + pad)

    # Remove all chrome
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_zlabel('')
    ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.pane.set_edgecolor('none')
        axis.line.set_color((1, 1, 1, 0))
    ax.grid(False)
    ax.view_init(elev=elev, azim=azim)

    # Render to in-memory PNG
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', pad_inches=0,
                transparent=False, facecolor='white')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert('RGB')


def _crop_whitespace(img, threshold=10):
    """Find the bounding box of non-white content. Returns (left, top, right, bottom)."""
    arr = np.array(img, dtype=np.int16)
    mask = np.abs(arr - 255).sum(axis=2) > threshold
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return cmin, rmin, cmax + 1, rmax + 1


def create_pose_row(poses, filename, output_dir, elev=10, azim=0):
    """Render each pose, crop, pad to uniform square cells, then stitch."""
    # 1. Render all raw images (same matplotlib figure size -> same pixel size)
    raw_imgs = [_render_single_pose(p, elev=elev, azim=azim) for p in poses]

    # 2. Find content bounding box for each
    bboxes = [_crop_whitespace(im) for im in raw_imgs]

    # 3. Determine the largest content width and height across ALL poses
    widths  = [b[2] - b[0] for b in bboxes]
    heights = [b[3] - b[1] for b in bboxes]
    max_w = max(widths)
    max_h = max(heights)

    # Uniform cell size: use the larger of max_w, max_h + comfortable margin
    margin = int(max(max_w, max_h) * 0.12)
    cell_w = max_w + 2 * margin
    cell_h = max_h + 2 * margin

    # 4. Create each cell: crop content, center it in the uniform cell
    cells = []
    for im, (l, t, r, b) in zip(raw_imgs, bboxes):
        content = im.crop((l, t, r, b))
        cell = Image.new('RGB', (cell_w, cell_h), (255, 255, 255))
        offset_x = (cell_w - content.width) // 2
        offset_y = (cell_h - content.height) // 2
        cell.paste(content, (offset_x, offset_y))
        cells.append(cell)

    # 5. Stitch horizontally
    gap = 0
    total_w = cell_w * len(cells) + gap * (len(cells) - 1)
    canvas = Image.new('RGB', (total_w, cell_h), (255, 255, 255))
    for i, cell in enumerate(cells):
        canvas.paste(cell, (i * (cell_w + gap), 0))

    output_dir.mkdir(parents=True, exist_ok=True)
    canvas.save(output_dir / f"{filename}.png", dpi=(300, 300))
    canvas.save(output_dir / f"{filename}.pdf", dpi=(300, 300))
    print(f"Saved: {output_dir / filename} (.png, .pdf)  cell={cell_w}x{cell_h}  total={canvas.size}")

    # Display inline
    fig, ax_show = plt.subplots(1, 1, figsize=(16, 16 * cell_h / total_w))
    ax_show.imshow(canvas)
    ax_show.axis('off')
    fig.tight_layout(pad=0)
    plt.show()
    plt.close(fig)

In [ ]:
output_dir = Path("output")

create_pose_row(mmbody_poses, "mmbody_poses", output_dir)
create_pose_row(mmfi_poses, "mmfi_poses", output_dir)
create_pose_row(mmbody_uncovered_poses, "amass_uncovered_by_mmbody", output_dir)
create_pose_row(mmfi_uncovered_poses, "amass_uncovered_by_mmfi", output_dir)

print("All figures saved to:", output_dir.resolve())

In [ ]:
# (empty - all logic moved to cells above)